# Mixed Logit — Standalone Fit and Search

Mixed Logit (MXL) extends MNL by allowing coefficients to vary across
decision-makers according to a specified distribution. This captures
**taste heterogeneity** — e.g. different travellers may value travel
time savings differently.

The log-likelihood is approximated via simulation (Halton draws).
MLE is JAX-accelerated.

**Supported distributions:** normal (`n`), log-normal (`ln`),
triangular (`t`), truncated-normal (`tn`), uniform (`u`).

In [ ]:
!pip install SearchLibrium --upgrade -q

In [ ]:
import numpy as np
import pandas as pd
from SearchLibrium import MixedLogit, Parameters, call_siman

## 1. Load data

In [ ]:
url = 'https://raw.githubusercontent.com/zahern/HypothesisX/refs/heads/main/data/Swissmetro_final.csv'
df  = pd.read_csv(url)
print(df.columns.tolist())
print(df.head())

## 2. Standalone Mixed Logit fit

We fix TIME and COST as normally-distributed random parameters.

In [ ]:
varnames   = ['TIME', 'COST', 'HEADWAY', 'SEATS']
choice_set = np.unique(df['alt']).tolist()

mxl = MixedLogit()
mxl.setup(
    X        = df[varnames],
    y        = df['CHOICE'].values,
    varnames = varnames,
    alts     = df['alt'].values,
    ids      = df['custom_id'].values,
    panels   = df['ID'].values,
    randvars = {'TIME': 'n', 'COST': 'ln'},   # TIME~normal, COST~lognormal
    n_draws  = 500,
    base_alt = 'SM',
)
mxl.fit()
mxl.summarise()

## 3. Search over Mixed Logit specifications

The search simultaneously explores:
- Which variables to include
- Which variables should be random, and with which distribution
- Whether to add Box-Cox transformations

In [ ]:
params = Parameters(
    criterions   = [('bic', -1)],
    df           = df,
    varnames     = varnames,
    asvarnames   = varnames,
    isvarnames   = [],
    choice_set   = choice_set,
    choices      = df['CHOICE'].values,
    alt_var      = df['alt'].values,
    choice_id    = df['custom_id'].values,
    ind_id       = df['ID'].values,
    base_alt     = 'SM',
    models       = ['mixed_logit'],
    allow_random = True,        # REQUIRED — enables random parameter search
    allow_bcvars = True,
    n_draws      = 500,
    p_val        = 0.05,
    all_sig      = True,
)

best = call_siman(params, init_sol=None, id_num=1,
                  ctrl=(200, 0.001, 50, 10))

## 4. Inspect the best model

In [ ]:
if best and best.get('model'):
    best['model'].summarise()

print('Variables:     ', best.get('asvars'))
print('Random params: ', best.get('randvars'))
print('BIC:           ', best.get('bic'))
print('Log-lik:       ', best.get('loglik'))